## Ejercicio: Trabajador de conocimiento basado en Inteligencia Artificial usando RAG

1. Crea un trabajador del conocimiento en tus datos para impulsar la productividad

* Reúne todos tus archivos en un solo lugar: tu Base de Conocimientos personal
* Vectoriza todo en Chroma, tu almacen de vectores
* Crea un asistente de IA conversacional y formula preguntas

2. Ideas avanzadas para llevarlo al siguiente nivel:

* Si utilizas Google Workspace, usa la API de Google para leer tus propios documentos
* Si utilizas MS Office, usa biblioteca para leer documentos de Office
* Má difícil: usa biblioteca para conectarte a tu bandeja de entrada de coreo electrónico, Slack y más

3. Opciones para mejorar la seguridad:

* Puedes usar un modelos de código abierto
* Crear la función de embbeddings
* Investigar .cp, librería que se ejecuta en local sin tener que conectar a Internet, vectorizar los documentos si tener que ir a la nube

4. Requisitos mínimos del projecto:

* Carpeta con algunos documentos pdf
* Versión en miniatura con algunos documentos de texto

## Solución:

Se crea un trabajador de conocimiento de mis datos en una carpeta local para impulsar la productividad. Se usan guías en pdf de buenas prácticas de seguridad en el ámbito de AI Security y Zero Trust.

1. Se crea la carpeta exercise-knowledge-base/ y se añaden dos carpetas con documentos:
    * ai-security/ con documentos pdf de OWASP sobre amenazas, riesgos y controles de seguridad que afectan a las aplicaciones de IA.
    * zero-trust/ con documentos pdf relacionados con el enfoque de seguridad Zero-Trust.
2. Se resuelven algunos problemas con las librerias ya que no se encuentran algunos módulos:
    * DirectoryLoader se encuentra en langchain_community.document_loaders hay que instalar langchain-community
    * Document se encuentra en langchain_core.documents hay que instalar langchain-core
    * CoversationBufferMemory se encuentra en langchain_classic.memory
    * ConversationalRetrievalChain se encuentra en langchain_classic.chains
3. Al ser documentos pdf tenemos que importar el módulo PyPDFLoader de la librería langchain_community.document_loaders para leer los documentos.
4. Se trocea el contenido usando CharacterTextSplitter.
5. Se leen los documentos. Asignamos cada fragmento de texto a un vector usando OpenAIEmbeddings y se crea el almacén de vectores con Chroma. Se encuentran dos tipos de documentos correspondientes a las dos carpetas: ai-security y zero-trust. Se identifican 965 vectores con 1536 dimensiones en el almacén de vectores.
6. Se visualiza el almacén de vectores.
    * Se puede ver, tanto en la representación en 2D como en 3D, la separación de los grupos embeddings creados a partir del contenido de los documentos de ambas carpetas y por tanto de su semantica.
    * Con respecto a los documentos de ai-security hay una separación entre la documentación de seguridad de OWASP y la de Microsoft, se puede ver agrupaciones de puntos separadas.
    * Los puntos que representan el contenido de zero-trust están más dispersos porque los documentos tocan el principio de zero-trust en muchas areas tecnológicas.
7. Se crea un chat con Langchain usando ConversationalRetrievalChain.
    * Se hacen preguntas concretas para comprobar que la solución está funcionando correctamente y que se están añadiendo los datos relacionados del contexto de la pregunta a la entrada del LLM.
    * Para comprobarlo se pregunta por segundo riesgo de seguridad más critico de las aplicaciones LLM.
    * Teniendo en cuenta que hay varias versiones publicadas del Top 10 de OWASP de LLMs, la proporcionada en los documentos pdf (de 2025) considera el segundo riesgo "Sensitive Information Disclosure", que es diferente a la versión publicada en 2023 que considera al "Insecure Output Handling" como el segundo más critico (versión no proporcionada en la carpeta exercise-knowledge-base)
    * Entonces, ante la pregunta: "Por favor, indica cuál es el segundo riesgo de seguridad más critico de las aplicaciones LLMs."
    * El sistema responde: "El segundo riesgo de seguridad más crítico de las aplicaciones LLM es la "Divulgación de Información Sensible" (LLM02:2025). Este riesgo se refiere a la posibilidad de que información sensible, como datos personales identificables (PII), detalles financieros, registros de salud, datos confidenciales de negocios, credenciales de seguridad y documentos legales, sea expuesta a través de la salida del LLM. Esto puede resultar en acceso no autorizado a datos, violaciones de privacidad y violaciones de propiedad intelectual."
8. Se muestra en una interfaz de Gradio.
9. Se investiga que se envía detrás de la escena.



In [73]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr

In [74]:
# imports de langchain, plotly y Chroma

from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain

In [75]:
MODEL = "gpt-4o-mini"
db_name = "vector_db"

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [76]:
# Leer documentos usando los cargadores de LangChain
# Tomar todo lo que está en todas las subcarpetas de nuestra base de conocimiento

knowledge_base_folder = "exercise-knowledge-base/*"

folders = glob.glob(knowledge_base_folder)

def add_metadata(doc, doc_type):
    doc.metadata["doc_type"] = doc_type
    return doc

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
    )
    folder_docs = loader.load()
    documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

print(f"Total number of chunks: {len(chunks)}")
print(f"Document types found: {set(doc.metadata['doc_type'] for doc in documents)}")

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 64 0 (offset 0)
Ignoring wrong pointing object 75 0 (offset 0)
Ignoring wrong pointing object 77 0 (offset 0)
Ignoring wrong pointing object 180 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 149 0 (offset 0)
Ignoring wrong pointing object 257 0 (offset 0)
Ignoring wrong pointing object 320 0 (offset 0)


Total number of chunks: 965
Document types found: {'ai-security', 'zero-trust'}


## Una nota al margen sobre las embeddings y los "LLM de codificación automática"

Asignaremos cada fragmento de texto a un vector que representa el significado del texto, conocido como incrustación.

OpenAI ofrece un modelo para hacer esto, que utilizaremos llamando a su API con un código LangChain.

Este modelo es un ejemplo de un "LLM de codificación automática" que genera una salida dada una entrada completa.
Es diferente a todos los demás LLM que hemos analizado hoy, que se conocen como "LLM autorregresivos", y generan tokens futuros basados ​​solo en el contexto pasado.

Otro ejemplo de un LLM de codificación automática es BERT de Google. Además de la incrustación, los LLM de codificación automática se utilizan a menudo para la clasificación.

### Nota al margen

En la semana 8 volveremos a RAG y a las incrustaciones vectoriales, y utilizaremos un codificador vectorial de código abierto para que los datos nunca abandonen nuestra computadora; esa es una consideración importante cuando se crean sistemas empresariales y los datos deben permanecer internos.

In [77]:
# Coloque los fragmentos de datos en un almacén de vectores que asocie una incrustación de vectores con cada fragmento
# Chroma es una popular base de datos de vectores de código abierto basada en SQLLite

embeddings = OpenAIEmbeddings()

# Eliminar si ya existe

if os.path.exists(db_name):
    Chroma(
        persist_directory=db_name,
        embedding_function=embeddings
    ).delete_collection()

# Crear almacén de vectores

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 965 documents


In [79]:
# Investiguemos los vectores

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"Hay {count:,} vectores con {dimensions:,} dimensiones en el almacén de vectores")

Hay 965 vectores con 1,536 dimensiones en el almacén de vectores


## Visualización del almacén de vectores

Tomémonos un minuto para observar los documentos y sus vectores de incrustación para ver qué está sucediendo.

In [80]:
# Trabajo previo (¡con agradecimiento a Jon R por identificar y corregir un error en esto!)

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
#colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]
colors = [['blue', 'green'][['ai-security', 'zero-trust'].index(t)] for t in doc_types]

In [81]:
# ¡A los humanos nos resulta más fácil visualizar cosas en 2D!
# Reducir la dimensionalidad de los vectores a 2D usando t-SNE
# (incrustación de vecinos estocásticos distribuidos en t)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(
        size=5,
        color=colors,
        opacity=0.8
    ),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='Visualización 2D Chroma Vector Store',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [82]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(
        size=5,
        color=colors,
        opacity=0.8
    ),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='Visualización 3D Chroma Vector Store',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## Es hora de usar LangChain para unirlo todo

In [83]:
# create a new Chat with OpenAI
llm = ChatOpenAI(
    temperature=0.7,
    model_name=MODEL
)

# set up the conversation memory for the chat
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True
)

# the retriever is an abstraction over the VectorStore that will be used during RAG
retriever = vectorstore.as_retriever()

# juntando todo: configure la cadena de conversación con GPT 3.5 LLM, el almacén vectorial y la memoria
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

In [84]:
# Vamos a intentar una pregunta sencilla

query = "Por favor, indica cuál es el segundo riesgo de seguridad más critico de las aplicaciones LLMs."
result = conversation_chain.invoke({"question": query})
print(result["answer"])

El segundo riesgo de seguridad más crítico de las aplicaciones LLM es la "Divulgación de Información Sensible" (LLM02:2025). Este riesgo se refiere a la posibilidad de que información sensible, como datos personales identificables (PII), detalles financieros, registros de salud, datos confidenciales de negocios, credenciales de seguridad y documentos legales, sea expuesta a través de la salida del LLM. Esto puede resultar en acceso no autorizado a datos, violaciones de privacidad y violaciones de propiedad intelectual.


In [85]:
# set up a new conversation memory for the chat
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True
)

# putting it together: set up the conversation chain with the GPT 4o-mini LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

## Ahora, lo mostraremos en Gradio usando la interfaz de Chat:

Una forma rápida y sencilla de crear un prototipo de chat con un LLM

In [86]:
# Wrapping that in a function

def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

In [87]:
# And in Gradio:
print(f"Versión de Gradio: {gr.__version__}")

view = gr.ChatInterface(
    chat,
#    type="messages"
).launch(inbrowser=True)

Versión de Gradio: 6.3.0
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [89]:
# Investiguemos qué se envía detrás de escena

from langchain_core.callbacks import StdOutCallbackHandler

llm = ChatOpenAI(
    temperature=0.7,
    model_name=MODEL
)

memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True
)

retriever = vectorstore.as_retriever()

conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    callbacks=[StdOutCallbackHandler()]
)

query = "Por favor, indica cuál es el segundo riesgo de seguridad más critico de las aplicaciones LLMs."
result = conversation_chain.invoke({"question": query})
answer = result["answer"]
print("\nRespuesta:", answer)



> Entering new ConversationalRetrievalChain chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
OWASP Top 10 for LLM Applications v2.0
11genai.owasp.org
LLM03:2025 Supply Chain
Description
LLM supply chains are susceptible to various vulnerabilities, which can affect the integrity of
training data, models, and deployment platforms. These risks can result in biased outputs,
security breaches, or system failures. While traditional software vulnerabilities focus on issues
like code flaws and dependencies, in ML the risks also extend to third-party pre-trained models
and data.
These external elements can be manipulated through tampering or poisoning attacks.
Creating LLMs is a specialized task that often depends on third-party models. The rise o

In [90]:
# crear un nuevo Chat con OpenAI
llm = ChatOpenAI(
    temperature=0.7,
    model_name=MODEL
)

# Configurar la memoria de conversación para el chat.
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True
)

# el recuperador es una abstracción sobre el VectorStore que se utilizará durante RAG; k es la cantidad de fragmentos a utilizar
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 60
    }
)

# juntando todo: configure la cadena de conversación con GPT 3.5 LLM, el almacén vectorial y la memoria
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

In [91]:
def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

In [92]:
view = gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
